# Hierarchical binomial model

Placeholder: introduce the model-fitting and parameterization comparison presented in this section.

## Setup

Placeholder: describe the software, dataset sizes, and reproducibility settings used below.

In [ ]:
from collections.abc import Sequence
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import plotnine as p9
from cmdstanpy import CmdStanMCMC, CmdStanModel

STAN_DIR = Path("stan")
IMAGE_DIR = Path("img")
IMAGE_DIR.mkdir(exist_ok=True)

N_GROUPS = 9
N_OBS = (2, 4, 8, 16, 64, 256, 2048, 32768)

p9.options.figure_size = (6, 6)
p9.options.dpi = 100
np.set_printoptions(precision=3)

## Simulate nested datasets

Placeholder: explain why every dataset is constructed from a prefix of the same simulated sequence.

In [ ]:
data_generator = CmdStanModel(
    stan_file=str(STAN_DIR / "datagen_repeated_binary_trials.stan")
)

simulation = data_generator.sample(
    data={"N_groups": N_GROUPS, "N_obs": max(N_OBS)},
    chains=1,
    iter_warmup=0,
    iter_sampling=1,
    adapt_engaged=False,
    show_progress=False,
    seed=12345,
)

In [ ]:
def make_nested_datasets(
    trials: np.ndarray, n_obs_values: Sequence[int]
) -> dict[int, dict[str, object]]:
    """Create Stan datasets from cumulative success counts."""
    cumulative_successes = np.cumsum(trials, axis=0)
    return {
        n_obs: {
            "N": n_obs,
            "y": cumulative_successes[n_obs - 1].tolist(),
        }
        for n_obs in n_obs_values
    }

In [ ]:
theta_true = simulation.stan_variable("theta")[0]
trials = simulation.stan_variable("y")[0].astype(int)
datasets = make_nested_datasets(trials, N_OBS)
print(f'theta {theta_true}')
pd.DataFrame(
    {
        "N": N_OBS,
        "successes by group": [datasets[n_obs]["y"] for n_obs in N_OBS],
    }
)

## Compile the centered and non-centered models

Placeholder: connect each Stan program to the parameter coordinates sampled by Hamiltonian Monte Carlo.

In [ ]:
@dataclass(frozen=True)
class Parameterization:
    label: str
    model: CmdStanModel
    sampled_x: str


@dataclass(frozen=True)
class ModelFit:
    parameterization: Parameterization
    n_obs: int
    fit: CmdStanMCMC

In [ ]:
centered_model = CmdStanModel(
    stan_file=str(STAN_DIR / "funnel_data_cp.stan")
)
noncentered_model = CmdStanModel(
    stan_file=str(STAN_DIR / "funnel_data_ncp.stan")
)

PARAMETERIZATIONS = (
    Parameterization(
        label="centered",
        model=centered_model,
        sampled_x="theta[1]",
    ),
    Parameterization(
        label="non-centered",
        model=noncentered_model,
        sampled_x="theta_raw[1]",
    ),
)

## Fit every model to every dataset

Placeholder: describe the two-by-eight grid of fits and the sampling settings chosen for the comparison.

In [ ]:
def fit_dataset(
    parameterization: Parameterization,
    dataset: dict[str, object],
    seed: int,
) -> CmdStanMCMC:
    """Fit one parameterization to one Stan dataset."""
    return parameterization.model.sample(
        data=dataset,
        seed=seed,
        show_progress=False,
    )

In [ ]:
fits: list[ModelFit] = []

for parameterization in PARAMETERIZATIONS:
    for n_obs in N_OBS:
        fit = fit_dataset(
            parameterization=parameterization,
            dataset=datasets[n_obs],
            seed=12345 + n_obs,
        )
        fits.append(
            ModelFit(
                parameterization=parameterization,
                n_obs=n_obs,
                fit=fit,
            )
        )

## Summarize divergent transitions

Placeholder: explain how divergences reveal difficulty exploring the posterior geometry.

In [ ]:
def divergence_summary(fits: Sequence[ModelFit]) -> pd.DataFrame:
    """Return one row of divergence diagnostics per fitted model."""
    rows: list[dict[str, object]] = []

    for result in fits:
        divergent = result.fit.method_variables()["divergent__"]
        rows.append(
            {
                "parameterization": result.parameterization.label,
                "N": result.n_obs,
                "draws": divergent.size,
                "divergent": divergent.sum(),
                "percent divergent": 100 * divergent.mean(),
            }
        )

    summary = pd.DataFrame(rows)
    parameterization_order = list(
        dict.fromkeys(result.parameterization.label for result in fits)
    )
    summary["parameterization"] = pd.Categorical(
        summary["parameterization"],
        categories=parameterization_order,
        ordered=True,
    )
    return summary.sort_values(["parameterization", "N"], ignore_index=True)

In [ ]:
divergence_summary(fits)

## Plot the parameter space sampled by each model

Placeholder: interpret the centered coordinates `(theta[1], sigma)` and non-centered coordinates `(theta_raw[1], sigma)`.

In [ ]:
def draws_xy(
    fit: CmdStanMCMC, x_variable: str, y_variable: str
) -> pd.DataFrame:
    """Extract two variables and the divergence indicator from a fit."""
    draws = fit.draws_pd()
    return pd.DataFrame(
        {
            "x": draws[x_variable].to_numpy(),
            "y": draws[y_variable].to_numpy(),
            "divergent": draws["divergent__"].to_numpy() > 0,
        }
    )

In [ ]:
def facet_frame(
    fits: Sequence[ModelFit],
    n_obs_values: Sequence[int],
    y_variable: str = "log_sigma_sq",
) -> pd.DataFrame:
    """Stack selected fits into a tidy frame for faceted plotting."""
    requested_sizes = set(n_obs_values)
    selected_fits = [result for result in fits if result.n_obs in requested_sizes]

    frames = []
    for result in selected_fits:
        parameterization = result.parameterization
        facet_label = f"{parameterization.label}: {parameterization.sampled_x}"
        frames.append(
            draws_xy(
                result.fit,
                x_variable=parameterization.sampled_x,
                y_variable=y_variable,
            ).assign(
                parameterization=facet_label,
                N=f"N = {result.n_obs}",
            )
        )

    frame = pd.concat(frames, ignore_index=True)
    parameterization_order = list(
        dict.fromkeys(
            f"{result.parameterization.label}: {result.parameterization.sampled_x}"
            for result in fits
        )
    )
    frame["parameterization"] = pd.Categorical(
        frame["parameterization"],
        categories=parameterization_order,
        ordered=True,
    )
    frame["N"] = pd.Categorical(
        frame["N"],
        categories=[f"N = {n_obs}" for n_obs in n_obs_values],
        ordered=True,
    )
    return frame

In [ ]:
def plot_divergence_grid(
    data: pd.DataFrame,
    title: str,
    subtitle: str = "Divergent transitions in red",
    figure_size: tuple[float, float] = (14.0, 7.0),
    coords: p9.coord_cartesian | None = None,
) -> p9.ggplot:
    """Plot sampler coordinates using fixed grid size."""
    nondivergent = data[~data["divergent"]]
    divergent = data[data["divergent"]]
    if coords is None:
      coords = p9.coord_cartesian(xlim=(-10, 10), ylim=(-10, 10))

    return (
        p9.ggplot(data, p9.aes(x="x", y="y"))
        + p9.geom_point(
            data=nondivergent, color="#333333", alpha=0.4, size=0.7
        )
        + p9.geom_point(data=divergent, color="red", alpha=0.8, size=0.7)

        + p9.facet_grid(
          rows="parameterization",
          cols="N",
          scales="fixed",
          )
        + coords
        + p9.labs(
            x="group 1 parameter",
            y="log_sigma_sq",
            title=title,
            subtitle=subtitle,
        )
        + p9.theme_minimal()
        + p9.theme(figure_size=figure_size)
    )

In [ ]:
print(f'data generating parameter theta[1]: {theta_true[0]}')

In [ ]:
LOW_DATA_N = N_OBS[:4]

low_data_plot = plot_divergence_grid(
    facet_frame(fits, LOW_DATA_N),
    title="Low data regime: group[1] parameter against population scale",
    coords = p9.coord_cartesian(xlim=(-4, 8), ylim=(-12, 6))
)
low_data_plot.save(
    IMAGE_DIR / "funnel_cp_ncp_low_data.png",
    dpi=150,
    verbose=False,
)
low_data_plot

In [ ]:
HIGH_DATA_N = N_OBS[4:]
high_data_plot = plot_divergence_grid(
    facet_frame(fits, HIGH_DATA_N),
    title="High data regime: sampler parameter against population scale",
    coords = p9.coord_cartesian(xlim=(-3, 3), ylim=(-3, 3))
)
high_data_plot.save(
    IMAGE_DIR / "funnel_cp_ncp_high_data.png",
    dpi=150,
    verbose=False,
)
high_data_plot